In [1]:
pip install pymupdf


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import fitz  # PyMuPDF
import json
import re

def clean_text(text):
    # Menghapus whitespace berlebih
    text = re.sub(r'\s+', ' ', text)
    # Menghapus header yang berulang (contoh: Kepala Desa Karangdoro)
    text = text.replace("KEPALA DESA KARANGDORO KECAMATAN TEGALSARI KABUPATEN BANYUWANGI", "")
    return text.strip()

def extract_perdes_to_jsonl(pdf_path, output_file):
    doc = fitz.open(pdf_path)
    dataset = []

    full_text = ""
    for page in doc:
        full_text += page.get_text("text") + "\n"

    # Pisahkan berdasarkan Pasal untuk segmentasi data hukum yang logis
    sections = re.split(r'(Pasal \d+)', full_text)
    
    # Bagian awal (sebelum Pasal 1) biasanya berisi Menimbang & Mengingat
    header_info = clean_text(sections[0])
    
    # Proses setiap Pasal
    for i in range(1, len(sections), 2):
        pasal_title = sections[i]
        pasal_content = clean_text(sections[i+1]) if i+1 < len(sections) else ""
        
        # Gabungkan konteks agar model tahu ini peraturan apa
        context = f"Peraturan Desa Karangdoro No 1 Tahun 2024 tentang LPJ APBDes 2023. {pasal_title}: {pasal_content}"
        
        data_point = {
            "instruction": f"Jelaskan isi dari {pasal_title} pada Perdes Karangdoro No 1 Tahun 2024.",
            "context": context,
            "response": pasal_content
        }
        dataset.append(data_point)

    # Simpan ke format JSONL
    with open(output_file, 'w', encoding='utf-8') as f:
        for entry in dataset:
            f.write(json.dumps(entry) + '\n')
            
    print(f"Berhasil mengekstrak {len(dataset)} bagian ke {output_file}")

# Jalankan script
extract_perdes_to_jsonl("/home/oppir/TA-OPH/25-26/TA-07/app/data/2_PERDES_NO_1_2024.pdf", "/home/oppir/TA-OPH/25-26/TA-07/app/data/dataset_perdes.jsonl")

Berhasil mengekstrak 4 bagian ke /home/oppir/TA-OPH/25-26/TA-07/app/data/dataset_perdes.jsonl


BAGI